### Build Driver Standings 

In [0]:
SELECT *
FROM formula1.gold.fact_session_results
LIMIT 10;

SELECT *
FROM formula1.gold.dim_drivers
LIMIT 10;

round,season,constructor_id,driver_id,grid_position,completed_laps,car_number,points,final_position,final_position_text,status,session_type,is_win,is_podium,has_points
1,1950,alfa,farina,1,70,2,9.0,1,1,Finished,RACE,true,true,true
3,1950,wetteroth,rathmann,28,122,76,0.0,24,24,+16 Laps,RACE,false,false,false
3,1950,kurtis_kraft,agabashian,2,64,28,0.0,28,R,Oil leak,RACE,false,false,false
4,1950,maserati,bira,8,40,30,3.0,4,4,+2 Laps,RACE,false,false,true
5,1950,maserati,branca,11,29,30,0.0,10,10,+6 Laps,RACE,false,false,false
6,1950,lago,pozzi,15,56,26,0.0,6,6,+8 Laps,RACE,false,false,false
7,1950,alfa,taruffi,7,34,60,0.0,13,R,Engine,RACE,false,false,false
1,1951,ferrari,taruffi,6,42,44,6.0,2,2,Finished,RACE,false,true,true
1,1951,alfa,sanesi,4,41,28,3.0,4,4,+1 Lap,RACE,false,false,true
1,1951,lago,etancelin,12,39,4,0.0,10,10,+3 Laps,RACE,false,false,false


driver_id,driver_name,date_of_birth,nationality,nationality_region
barber,Skip Barber,1936-11-16,American,North America
bellof,Stefan Bellof,1957-11-20,German,Europe
belso,Tom Belsø,1942-08-27,Danish,Europe
bettenhausen,Tony Bettenhausen,1916-09-12,American,North America
biondetti,Clemente Biondetti,1898-08-18,Italian,Europe
brancatelli,Gianfranco Brancatelli,1950-01-18,Italian,Europe
chaboud,Eugène Chaboud,1907-04-12,French,Europe
chimeri,Ettore Chimeri,1921-06-04,Venezuelan,South America
comas,Érik Comas,1963-09-28,French,Europe
elisian,Ed Elisian,1926-12-09,American,North America


**Requirements**
Creating view that supports Drivers standings and total_points, shows race_starts, number_of_wins, number_of_podium. 
drivers recods should be grouped by season, driver_id, driver_name, driver_nationality, 
Ranked () Partition by season and  order by total_points, number_of_wins

In [0]:
CREATE OR REPLACE VIEW formula1.gold.v_driver_standings AS 
WITH dirver_session_summery AS (
  SELECT
    r.season,
    d.driver_id,
    d.driver_name,
    d.nationality,
    COUNT(*) AS race_starts,
    SUM(r.points) AS total_points,
    count_if(r.is_win) as number_of_wins,
    count_if(r.is_podium) as number_of_podium
  FROM
    formula1.gold.fact_session_results as r
      JOIN formula1.gold.dim_drivers as d
        ON r.driver_id = d.driver_id
  GROUP BY
    r.season,
    d.driver_id,
    d.driver_name,
    d.nationality
)
SELECT
  season,
  driver_id,
  driver_name,
  nationality,
  rank() OVER (PARTITION BY season ORDER BY total_points DESC, number_of_wins DESC) AS Standing,
  race_starts,
  total_points,
  number_of_wins,
  number_of_podium
FROM
  dirver_session_summery

In [0]:
SELECT driver_name, Standing FROM formula1.gold.v_driver_standings WHERE season = 2021 ORDER BY Standing

driver_name,Standing
Max Verstappen,1
Lewis Hamilton,2
Valtteri Bottas,3
Sergio Pérez,4
Carlos Sainz,5
Lando Norris,6
Charles Leclerc,7
Daniel Ricciardo,8
Pierre Gasly,9
Fernando Alonso,10
